# Build Processed Real Video Dataset

本文件是 processed dataset 构建入口。把 raw video dataset 转换为可消费的 processed dataset。

## 00 Runtime Identity And User Config

`REPO_URL` 与 `REPO_ROOT` 控制 Colab 冷启动时的仓库 bootstrap；
`RAW_DATASET_DOWNLOAD_MANIFEST_PATH`：指向 Drive 中的 raw dataset 注册表；
`RAW_DATASET_KEY` 选择注册表中的数据集条目；
`RAW_ARCHIVE_PATH` 指向原始压缩包；
`PROCESSED_DATASET_KEY` 冻结本次 processed dataset 的语义身份；
`PROCESSED_DATASET_ROOT`、`PROCESSED_DATASET_MANIFEST`、`PROCESSED_DATASET_CHECKS` 定义下游 notebook 会读取的交接文件。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = os.environ.get('TSTW_REPO_URL', 'https://github.com/RICHAAARC/TSTW-VW.git')
REPO_BRANCH = os.environ.get('TSTW_REPO_BRANCH', 'explicit-sync')
REPO_ROOT = Path(os.environ.get('TSTW_REPO_ROOT', '/content/TSTW-VW'))
REPO_PACKAGE_MARKER = REPO_ROOT / 'paper_workflow' / '__init__.py'
if not REPO_PACKAGE_MARKER.exists():
    if REPO_ROOT.exists():
        raise RuntimeError(f'Repository root exists but is incomplete: {REPO_ROOT}')
    clone_result = subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            '--branch',
            REPO_BRANCH,
            REPO_URL,
            str(REPO_ROOT),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            'repository bootstrap failed; set TSTW_REPO_URL to an authenticated clone URL or pre-clone the repository into TSTW_REPO_ROOT before running the notebook'
        )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from paper_workflow.notebook_utils import processed_real_video_dataset_workflow as dataset_workflow

WORKFLOW_KEY = 'processed_dataset_build'
RAW_DATASET_DOWNLOAD_MANIFEST_PATH = Path('/content/drive/MyDrive/Datasets/raw_dataset_download_manifest.json')
RAW_DATASET_KEY = 'davis_2017'
RAW_ARCHIVE_PATH = Path('/content/drive/MyDrive/Datasets/DAVIS_2017/raw/DAVIS-2017-trainval-480p.zip')
DRIVE_PROCESSED_ROOT = Path('/content/drive/MyDrive/TSTW/datasets/processed')
LOCAL_WORKSPACE_ROOT = Path('/content/TSTW_runtime/raw_workspaces/build_processed_real_video_dataset')
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe_davis2017_trainval480p_256x256_32f_8fps_freeze001'
PROCESSED_DATASET_ROOT = DRIVE_PROCESSED_ROOT / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = PROCESSED_DATASET_ROOT / 'dataset_manifest.json'
PROCESSED_DATASET_SUMMARY = PROCESSED_DATASET_ROOT / 'processed_dataset_summary.json'
PROCESSED_DATASET_CHECKS = PROCESSED_DATASET_ROOT / 'checks' / 'processed_dataset_checks.json'

## 01 Mount Google Drive

挂载 Google Drive, 并将 processed dataset handoff 写入 Drive 中的受治理数据目录。

挂载点固定为 `/content/drive`，后续 Drive 路径通过第 00 步定义的 `Path` 对象派生。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 02 Validate Raw Dataset Registry

确认 raw dataset 下载注册表存在并可用 UTF-8 读取。

`RAW_DATASET_DOWNLOAD_MANIFEST_PATH` 必须指向一个可读取的 JSON 文件。

`raw_dataset_registry` 会保存在 notebook session 中，后续按 `RAW_DATASET_KEY` 选择条目。


In [ ]:
if not RAW_DATASET_DOWNLOAD_MANIFEST_PATH.exists():
    raise FileNotFoundError(RAW_DATASET_DOWNLOAD_MANIFEST_PATH)
raw_dataset_registry = json.loads(RAW_DATASET_DOWNLOAD_MANIFEST_PATH.read_text(encoding='utf-8'))
print({'raw_dataset_download_manifest.json': str(RAW_DATASET_DOWNLOAD_MANIFEST_PATH)})

## 03 Select Raw Dataset Source

`RAW_DATASET_KEY` 选择条目，并打印 raw archive 路径与注册表命中状态。

`RAW_DATASET_KEY` 应与数据集 key 一致；`RAW_ARCHIVE_PATH` 应指向该数据集对应的压缩包。

In [ ]:
raw_dataset_source = raw_dataset_registry.get('datasets', {}).get(RAW_DATASET_KEY, {})
print({
    'raw_dataset_key': RAW_DATASET_KEY,
    'raw_archive_path': str(RAW_ARCHIVE_PATH),
    'registry_entry_present': bool(raw_dataset_source),
})

## 04 Prepare Local Dataset Workspace

创建 Colab session local workspace, 作为 raw archive 解压与视频片段处理的临时工作目录。

如果需要重新构建同一 processed dataset, 可以保持 Drive 输出目录不变, 只清理或更换本地 workspace。


In [ ]:
LOCAL_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
print({'local_workspace_root': str(LOCAL_WORKSPACE_ROOT)})

## 05 Extract Raw Dataset Archive

确认 raw archive 与本地解压 workspace。

`RAW_ARCHIVE_PATH` 是输入压缩包；`LOCAL_WORKSPACE_ROOT` 是解压和临时处理目录。

In [ ]:
print({
    'raw_archive_path': str(RAW_ARCHIVE_PATH),
    'extract_workspace': str(LOCAL_WORKSPACE_ROOT),
})

## 06 Build Processed Video Clips

数据解压、视频片段标准化、processed dataset manifest 写出、summary 写出与 checks 写出。

`processed_dataset_root` 为 Drive 中的 processed dataset 输出目录；
`processed_dataset_key` 为 registry 与下游 notebook 使用的稳定身份；
`clean_workspace=False` 表示保留当前 workspace, 便于调试重复运行。

In [ ]:
processed_dataset_handoff = dataset_workflow.build_processed_dataset_handoff(
    raw_dataset_download_manifest_path=RAW_DATASET_DOWNLOAD_MANIFEST_PATH,
    raw_dataset_key=RAW_DATASET_KEY,
    raw_archive_path=RAW_ARCHIVE_PATH,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_workspace_root=LOCAL_WORKSPACE_ROOT,
    clean_workspace=False,
)
print(processed_dataset_handoff)

## 07 Write Processed Dataset Manifest

读取并展示确认 `dataset_manifest.json`, 已经生成正式数据 handoff。

In [ ]:
processed_dataset_manifest = json.loads(PROCESSED_DATASET_MANIFEST.read_text(encoding='utf-8'))
print({
    'dataset_name': processed_dataset_manifest['dataset_name'],
    'PROCESSED_DATASET_MANIFEST': str(PROCESSED_DATASET_MANIFEST),
})

## 08 Validate Processed Dataset

读取 processed dataset checks 并要求其 `status` 为真, 用于阻断缺失视频、空 manifest 或处理失败的数据集进入下游运行。

In [ ]:
processed_dataset_checks = json.loads(PROCESSED_DATASET_CHECKS.read_text(encoding='utf-8'))
if not processed_dataset_checks.get('status'):
    raise RuntimeError(processed_dataset_checks)
print({'processed_dataset_checks.json': str(PROCESSED_DATASET_CHECKS), 'status': processed_dataset_checks['status']})

## 09 Register Processed Dataset

展示 processed dataset registry 路径, 用于确认本次构建结果已经进入 Drive 侧数据登记表。

In [ ]:
print({'dataset_registry.json': processed_dataset_handoff['dataset_registry.json']})

## 10 Print Processed Dataset Handoff

汇总下游 run notebook 必须复制的关键参数, 包括 `PROCESSED_DATASET_KEY`、`PROCESSED_DATASET_ROOT`、`PROCESSED_DATASET_MANIFEST`、summary 与 checks 路径。

`PROCESSED_DATASET_KEY` 应直接复制到 `run_real_video_vae_latent_probe.ipynb` 的第 00 步。

In [ ]:
handoff_summary = {
    'PROCESSED_DATASET_KEY': PROCESSED_DATASET_KEY,
    'PROCESSED_DATASET_ROOT': str(PROCESSED_DATASET_ROOT),
    'PROCESSED_DATASET_MANIFEST': str(PROCESSED_DATASET_MANIFEST),
    'processed_dataset_summary.json': processed_dataset_handoff['processed_dataset_summary.json'],
    'processed_dataset_checks.json': processed_dataset_handoff['processed_dataset_checks.json'],
}
print(handoff_summary)